In [1]:
"""
Qwen3.5-0.8B LoRA Fine-tuning (Unsloth)
RTX 6000 Pro Blackwell (96GB) / chi 환경
completion_only_loss 방식 (prompt-completion 데이터셋)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
MODEL_ID = "unsloth/Qwen3.5-9B"         # 27B → 9B
DATASET_PATH = "./data/7_qwen_dataset_train.jsonl"
OUTPUT_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split"  # 경로도 바꾸기

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 7e-5
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 모델 + 토크나이저
# -------------------------
print("📦 모델 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
# 또는 base_model 경유
print(model.base_model.config._attn_implementation)

# -------------------------
# LoRA
# -------------------------
print("🔧 LoRA 어댑터 부착...")
model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
)
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    """
    messages를 prompt(system+user)와 completion(assistant)으로 분리.
    prompt는 assistant turn 시작 마커까지 포함하고,
    completion은 순수 assistant 내용만.
    """
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    # prompt: system + user + "<|im_start|>assistant\n" 까지
    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,   # assistant 시작 마커까지 포함
    )

    # completion: assistant 본문 + "<|im_end|>\n"
    # (EOS가 반드시 학습되도록 명시)
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"

    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check: 첫 샘플 확인
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

# prompt가 assistant 마커로 끝나는지
assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
# completion이 EOS로 끝나는지
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,   # ← 핵심: completion 부분만 loss
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

#trainer.train(resume_from_checkpoint=True)
trainer.train()

# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

✅ 컴파일 캐시 삭제 확인 완료
✅ UNSLOTH_TARGET_GB = 4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth TARGET_GB = 4
📦 모델 로딩...
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.936 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 760/760 [00:01<00:00, 462.49it/s]


sdpa
sdpa
🔧 LoRA 어댑터 부착...
Unsloth: Making `model.base_model.model.model.language_model` require gradients
trainable params: 116,391,936 || all params: 9,526,205,680 || trainable%: 1.2218
📂 데이터셋 로딩...


...

✅ 데이터셋: 63080개 샘플
   컬럼: ['prompt', 'completion']

🔍 첫 샘플 검증
  prompt 끝부분:     ' 있을 것 같습니다\n[90.0-92.0] 이게 바로<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
  completion 앞부분: '{"instances":[{"text":"더맛있을까요?","start_sec":0.1,"end_sec":4.6},{"text":"냄비입니다","'
  completion 끝부분: 't_sec":91.9,"end_sec":92.1}]}<|im_end|>\n'
  ✅ prompt/completion 경계 OK

🔍 토큰 길이 필터링...
✅ 필터: 63080개 → 63080개 (0개 제외)
🚀 학습 시작...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 63,080 | Num Epochs = 1 | Total steps = 7,885
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 116,391,936 of 9,526,205,680 (1.22% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.181088
20,1.098221
30,1.103446
40,1.027641
50,0.890384
60,0.816969
70,0.791527
80,0.757802
90,0.808927
100,0.746909


💾 어댑터 저장...
✅ 완료! 저장 위치: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split


In [2]:
# %% LoRA 어댑터 재로드 → merge → 표준 토크나이저로 저장
import os
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel
from transformers import AutoTokenizer

BASE_MODEL     = "unsloth/Qwen3.5-9B"
LORA_DIR       = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

print("📦 LoRA 어댑터 로딩...")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print("🔧 토크나이저를 표준 HF 포맷으로 재저장...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=False)
for fn in [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "merges.txt",
    "added_tokens.json", "chat_template.jinja",
]:
    p = os.path.join(MERGED_DIR, fn)
    if os.path.exists(p):
        os.remove(p)
base_tok.save_pretrained(MERGED_DIR)

print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

📦 LoRA 어댑터 로딩...
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.936 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 760/760 [00:01<00:00, 468.95it/s] 


🔀 merge 중... → ./model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged
Found HuggingFace hub cache directory: /home/taeyoung/nfs-mount/chi2027/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]


Checking cache directory for required files...


Unsloth: Copying 4 files from cache to `./model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged`: 100%|██████████| 4/4 [00:37<00:00,  9.35s/it]


Successfully copied all 4 files from cache to `./model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:04<00:00, 16.13s/it]


Unsloth: Merge process complete. Saved to `/home/taeyoung/nfs-mount/chi2027/model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged`
🔧 토크나이저를 표준 HF 포맷으로 재저장...
✅ 완료: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split-merged
   파일 목록: ['.cache', 'chat_template.jinja', 'config.json', 'model.safetensors-00001-of-00004.safetensors', 'model.safetensors-00002-of-00004.safetensors', 'model.safetensors-00003-of-00004.safetensors', 'model.safetensors-00004-of-00004.safetensors', 'model.safetensors.index.json', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 (Unsloth)
기존 epoch 1 학습된 어댑터를 로드해서 추가 epoch 학습
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split"        # 기존 학습 결과
OUTPUT_DIR    = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep2"    # 새 저장 경로
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1          # 추가 1 epoch (누적 2 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 3e-5    # 기존 7e-5의 약 절반, 미세 조정용
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터를 붙인 모델 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,       # ← base가 아니라 기존 어댑터 경로
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

# 어댑터 이미 로드됨 → get_peft_model 호출하지 않음
# 그래디언트가 LoRA 파라미터에만 흐르는지 확인
model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"
    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)
print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 이어서 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,              # 0.05 → 0.01 (이미 데워진 상태)
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=43,                        # 기존과 다른 seed → 다른 순서로 데이터 노출
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

✅ 컴파일 캐시 삭제 확인 완료
✅ UNSLOTH_TARGET_GB = 4
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/taeyoung/miniconda3/envs/chi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth TARGET_GB = 4
📦 기존 LoRA 어댑터 로딩: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split
==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Workstation Edition. Num GPUs = 1. Max memory: 94.936 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 760/760 [00:01<00:00, 465.25it/s]


sdpa
sdpa
trainable params: 116,391,936 || all params: 9,526,205,680 || trainable%: 1.2218
📂 데이터셋 로딩...


...

Filter (num_proc=32): 100%|██████████| 63080/63080 [00:01<00:00, 46028.19 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ 필터: 63080개 → 63080개 (0개 제외)
🚀 이어서 학습 시작...


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=32): 100%|██████████| 63080/63080 [00:16<00:00, 3942.37 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 63,080 | Num Epochs = 1 | Total steps = 7,885
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 116,391,936 of 9,526,205,680 (1.22% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.648353
20,0.575173
30,0.627573
40,0.629144
50,0.577028
60,0.621771
70,0.624419
80,0.606393
90,0.667604
100,0.613798


In [ ]:
# %% ep2 LoRA merge
import os, glob, shutil, json
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel

LORA_DIR       = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep2"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

# 1. 어댑터 로드
print(f"📦 LoRA 어댑터 로딩: {LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

# 2. merge 저장
print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

# 3. 토크나이저를 HF 원본 캐시에서 직접 복사
print("🔧 토크나이저를 HF 원본 캐시에서 복사...")
SNAPSHOTS_DIR = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots"
snapshot_dirs = sorted(glob.glob(os.path.join(SNAPSHOTS_DIR, "*")))
assert snapshot_dirs, f"❌ HF 캐시에 Qwen/Qwen3.5-9B 스냅샷 없음: {SNAPSHOTS_DIR}"
SRC = snapshot_dirs[-1]
print(f"   SRC: {SRC}")

copied = 0
for fn in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
           "vocab.json", "merges.txt", "added_tokens.json", "chat_template.jinja"]:
    src_path = os.path.join(SRC, fn)
    dst_path = os.path.join(MERGED_DIR, fn)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"   ✅ {fn}")
        copied += 1

assert copied > 0, "❌ 복사된 파일 없음. SRC 확인"

# 4. 확인
with open(os.path.join(MERGED_DIR, "tokenizer_config.json"), encoding="utf-8") as f:
    cfg = json.load(f)
print(f"\n   tokenizer_class = {cfg.get('tokenizer_class')}")
print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")

In [ ]:
"""
Qwen3.5-9B LoRA 이어서 훈련 - Epoch 3 (ep2 → ep3)
"""

import shutil, os

shutil.rmtree("/tmp/torchinductor_taeyoung", ignore_errors=True)
shutil.rmtree(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache"), ignore_errors=True)

assert not os.path.exists("/tmp/torchinductor_taeyoung"), "inductor 캐시 삭제 실패"
assert not os.path.exists(os.path.expanduser("~/nfs-mount/chi2027/unsloth_compiled_cache")), "unsloth 캐시 삭제 실패"
print("✅ 컴파일 캐시 삭제 확인 완료")

os.environ["UNSLOTH_TARGET_GB"] = "4"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

assert os.environ.get("UNSLOTH_TARGET_GB") == "4"
print(f"✅ UNSLOTH_TARGET_GB = {os.environ['UNSLOTH_TARGET_GB']}")

from unsloth import FastModel

import unsloth_zoo.fused_losses.cross_entropy_loss as _ce_loss
_ce_loss.TARGET_GB = "4"
print(f"✅ unsloth TARGET_GB = {_ce_loss.TARGET_GB}")

import torch
from datasets import load_dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback


class ClearCacheCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()


# -------------------------
# Config
# -------------------------
PREV_LORA_DIR = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep2"     # ep2 결과
OUTPUT_DIR    = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3"     # 새 저장 경로
DATASET_PATH  = "./data/7_qwen_dataset_train.jsonl"

MAX_SEQ_LENGTH = 131072
NUM_EPOCHS = 1          # 추가 1 epoch (누적 3 epoch 효과)
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1.5e-5  # ep2의 3e-5 → 절반. 수렴 후반에 더 작게
LORA_RANK = 64
LORA_ALPHA = 128
SAVE_STEPS = 500
LOGGING_STEPS = 10


# -------------------------
# 기존 LoRA 어댑터 로드
# -------------------------
print(f"📦 기존 LoRA 어댑터 로딩: {PREV_LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=PREV_LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    use_gradient_checkpointing="unsloth",
)

print(model.config._attn_implementation)
print(model.base_model.config._attn_implementation)

model.print_trainable_parameters()


# -------------------------
# 데이터셋 로드 + prompt/completion 변환
# -------------------------
print("📂 데이터셋 로딩...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def to_prompt_completion(example):
    msgs = example["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]
    assert len(asst_msgs) == 1, f"assistant turn이 1개가 아님: {len(asst_msgs)}"

    prompt_text = _tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"
    return {"prompt": prompt_text, "completion": completion_text}


dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset.column_names,
    num_proc=32,
    desc="prompt/completion 변환",
)

# ep1(seed 42), ep2(seed 43)와 다른 순서로 섞기
dataset = dataset.shuffle(seed=44)

print(f"✅ 데이터셋: {len(dataset)}개 샘플")
print(f"   컬럼: {dataset.column_names}")


# -------------------------
# Sanity check
# -------------------------
print("\n🔍 첫 샘플 검증")
_s = dataset[0]
print(f"  prompt 끝부분:     {repr(_s['prompt'][-80:])}")
print(f"  completion 앞부분: {repr(_s['completion'][:80])}")
print(f"  completion 끝부분: {repr(_s['completion'][-40:])}")

assert _s["prompt"].rstrip().endswith("assistant") or "<|im_start|>assistant" in _s["prompt"][-50:], \
    f"❌ prompt가 assistant 시작 마커로 끝나지 않음: {repr(_s['prompt'][-80:])}"
assert _s["completion"].rstrip().endswith("<|im_end|>"), \
    f"❌ completion이 EOS로 끝나지 않음: {repr(_s['completion'][-40:])}"
print("  ✅ prompt/completion 경계 OK\n")


# -------------------------
# 토큰 길이 필터링
# -------------------------
print("🔍 토큰 길이 필터링...")


def count_tokens(example):
    p_ids = _tok.encode(example["prompt"], add_special_tokens=False)
    c_ids = _tok.encode(example["completion"], add_special_tokens=False)
    example["n_tokens"] = len(p_ids) + len(c_ids)
    return example


dataset = dataset.map(count_tokens, num_proc=32, desc="토큰 수 계산")
before = len(dataset)
dataset = dataset.filter(lambda x: x["n_tokens"] <= MAX_SEQ_LENGTH, num_proc=32)
dataset = dataset.remove_columns("n_tokens")
print(f"✅ 필터: {before}개 → {len(dataset)}개 ({before - len(dataset)}개 제외)")


# -------------------------
# 학습
# -------------------------
print("🚀 ep3 학습 시작...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.01,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=50,
    save_only_model=False,
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_num_proc=32,
    dataloader_num_workers=4,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="tensorboard",
    seed=44,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
    callbacks=[ClearCacheCallback()],
)

trainer.train()


# -------------------------
# 저장
# -------------------------
print("💾 어댑터 저장...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 완료! 저장 위치: {OUTPUT_DIR}")

In [ ]:
# %% ep3 LoRA merge
import os, glob, shutil, json
os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

from unsloth import FastModel

LORA_DIR       = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3"
MERGED_DIR     = LORA_DIR + "-merged"
MAX_SEQ_LENGTH = 131072

# 1. 어댑터 로드
print(f"📦 LoRA 어댑터 로딩: {LORA_DIR}")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

# 2. merge 저장
print(f"🔀 merge 중... → {MERGED_DIR}")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

# 3. 토크나이저를 HF 원본 캐시에서 직접 복사
print("🔧 토크나이저를 HF 원본 캐시에서 복사...")
SNAPSHOTS_DIR = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots"
snapshot_dirs = sorted(glob.glob(os.path.join(SNAPSHOTS_DIR, "*")))
assert snapshot_dirs, f"❌ HF 캐시에 Qwen/Qwen3.5-9B 스냅샷 없음: {SNAPSHOTS_DIR}"
SRC = snapshot_dirs[-1]
print(f"   SRC: {SRC}")

copied = 0
for fn in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
           "vocab.json", "merges.txt", "added_tokens.json", "chat_template.jinja"]:
    src_path = os.path.join(SRC, fn)
    dst_path = os.path.join(MERGED_DIR, fn)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"   ✅ {fn}")
        copied += 1

assert copied > 0, "❌ 복사된 파일 없음. SRC 확인"

# 4. 확인
with open(os.path.join(MERGED_DIR, "tokenizer_config.json"), encoding="utf-8") as f:
    cfg = json.load(f)
print(f"\n   tokenizer_class = {cfg.get('tokenizer_class')}")
print(f"✅ 완료: {MERGED_DIR}")
print(f"   파일 목록: {sorted(os.listdir(MERGED_DIR))}")